In [2]:
import pandas as pd
from pathlib import Path

In [3]:
DATA_DIR=Path(r"C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot\Data")
OUTPUT_DIR=Path(r"C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot\Tests")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
def load_data():

    payments = pd.read_csv(
        DATA_DIR / "payments.csv",
        parse_dates=["payment_date"]
    )

    settlements = pd.read_csv(
        DATA_DIR / "settlements.csv",
        parse_dates=["settlement_date"]
    )

    bank_records = pd.read_csv(
        DATA_DIR / "bank_records.csv",
        parse_dates=["credit_date"]
    )

    refunds = pd.read_csv(
        DATA_DIR / "refunds.csv",
        parse_dates=["refund_date"]
    )

    return payments, settlements, bank_records, refunds

In [5]:
import pandas as pd
from pathlib import Path


# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_DIR = Path(
    r"C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot"
)

DATA_DIR = PROJECT_DIR / "Data"
OUTPUT_DIR = PROJECT_DIR / "Output"

OUTPUT_DIR.mkdir(
    exist_ok=True
)


# ============================================================
# LOAD DATA
# ============================================================

def load_data():

    payments = pd.read_csv(
        DATA_DIR / "payments.csv",
        parse_dates=["payment_date"]
    )

    settlements = pd.read_csv(
        DATA_DIR / "settlements.csv",
        parse_dates=["settlement_date"]
    )

    bank_records = pd.read_csv(
        DATA_DIR / "bank_records.csv",
        parse_dates=["credit_date"]
    )

    refunds = pd.read_csv(
        DATA_DIR / "refunds.csv",
        parse_dates=["refund_date"]
    )

    return (
        payments,
        settlements,
        bank_records,
        refunds
    )


# ============================================================
# RECONCILIATION ENGINE
# ============================================================

def reconcile():

    payments, settlements, bank_records, refunds = load_data()

    results = []

    for _, payment in payments.iterrows():

        payment_id = payment["payment_id"]
        gross_amount = payment["gross_amount"]
        payment_date = payment["payment_date"]

        # ----------------------------------------------------
        # Default evidence values
        # ----------------------------------------------------

        settlement_id = None
        settlement_date = None

        fee = None
        tax = None

        expected_net_amount = None
        actual_settlement_amount = None

        bank_amount = None
        refund_amount = None

        delay_days = None

        # ----------------------------------------------------
        # 1. Find settlement
        # ----------------------------------------------------

        settlement = settlements[
            settlements["payment_id"] == payment_id
        ]

        if settlement.empty:

            results.append({

                "payment_id":
                    payment_id,

                "gross_amount":
                    gross_amount,

                "payment_date":
                    payment_date,

                "settlement_id":
                    settlement_id,

                "settlement_date":
                    settlement_date,

                "fee":
                    fee,

                "tax":
                    tax,

                "expected_net_amount":
                    expected_net_amount,

                "settlement_net_amount":
                    actual_settlement_amount,

                "bank_amount":
                    bank_amount,

                "refund_amount":
                    refund_amount,

                "delay_days":
                    delay_days,

                "status":
                    "EXCEPTION",

                "reason":
                    "MISSING_SETTLEMENT"
            })

            continue

        # ----------------------------------------------------
        # Settlement exists
        # ----------------------------------------------------

        settlement = settlement.iloc[0]

        settlement_id = settlement["settlement_id"]
        settlement_date = settlement["settlement_date"]

        fee = settlement["fee"]
        tax = settlement["tax"]

        actual_settlement_amount = (
            settlement["net_amount"]
        )

        # ----------------------------------------------------
        # Calculate expected settlement amount
        #
        # IMPORTANT:
        # This is calculated immediately after obtaining the
        # settlement, before any exception branch.
        # ----------------------------------------------------

        expected_net_amount = round(
            gross_amount
            - fee
            - tax,
            2
        )

        # ----------------------------------------------------
        # 2. Check settlement delay
        # ----------------------------------------------------

        delay_days = (
            settlement_date
            - payment_date
        ).days

        if delay_days > 3:

            results.append({

                "payment_id":
                    payment_id,

                "gross_amount":
                    gross_amount,

                "payment_date":
                    payment_date,

                "settlement_id":
                    settlement_id,

                "settlement_date":
                    settlement_date,

                "fee":
                    fee,

                "tax":
                    tax,

                "expected_net_amount":
                    expected_net_amount,

                "settlement_net_amount":
                    actual_settlement_amount,

                "bank_amount":
                    None,

                "refund_amount":
                    None,

                "delay_days":
                    delay_days,

                "status":
                    "EXCEPTION",

                "reason":
                    "SETTLEMENT_DELAY"
            })

            continue

        # ----------------------------------------------------
        # 3. Find bank record
        # ----------------------------------------------------

        bank_record = bank_records[
            bank_records["reference"]
            == settlement_id
        ]

        if bank_record.empty:

            results.append({

                "payment_id":
                    payment_id,

                "gross_amount":
                    gross_amount,

                "payment_date":
                    payment_date,

                "settlement_id":
                    settlement_id,

                "settlement_date":
                    settlement_date,

                "fee":
                    fee,

                "tax":
                    tax,

                "expected_net_amount":
                    expected_net_amount,

                "settlement_net_amount":
                    actual_settlement_amount,

                "bank_amount":
                    None,

                "refund_amount":
                    None,

                "delay_days":
                    delay_days,

                "status":
                    "EXCEPTION",

                "reason":
                    "MISSING_BANK_RECORD"
            })

            continue

        bank_record = bank_record.iloc[0]

        bank_amount = bank_record["amount"]

        # ----------------------------------------------------
        # 4. Check for refund
        # ----------------------------------------------------

        refund = refunds[
            refunds["payment_id"]
            == payment_id
        ]

        if not refund.empty:

            refund_amount = (
                refund["refund_amount"].sum()
            )

            results.append({

                "payment_id":
                    payment_id,

                "gross_amount":
                    gross_amount,

                "payment_date":
                    payment_date,

                "settlement_id":
                    settlement_id,

                "settlement_date":
                    settlement_date,

                "fee":
                    fee,

                "tax":
                    tax,

                "expected_net_amount":
                    expected_net_amount,

                "settlement_net_amount":
                    actual_settlement_amount,

                "bank_amount":
                    bank_amount,

                "refund_amount":
                    refund_amount,

                "delay_days":
                    delay_days,

                "status":
                    "EXCEPTION",

                "reason":
                    "REFUND_REQUIRES_INVESTIGATION"
            })

            continue

        # ----------------------------------------------------
        # 5. Validate settlement calculation
        # ----------------------------------------------------

        if expected_net_amount != actual_settlement_amount:

            results.append({

                "payment_id":
                    payment_id,

                "gross_amount":
                    gross_amount,

                "payment_date":
                    payment_date,

                "settlement_id":
                    settlement_id,

                "settlement_date":
                    settlement_date,

                "fee":
                    fee,

                "tax":
                    tax,

                "expected_net_amount":
                    expected_net_amount,

                "settlement_net_amount":
                    actual_settlement_amount,

                "bank_amount":
                    bank_amount,

                "refund_amount":
                    refund_amount,

                "delay_days":
                    delay_days,

                "status":
                    "EXCEPTION",

                "reason":
                    "SETTLEMENT_CALCULATION_MISMATCH"
            })

            continue

        # ----------------------------------------------------
        # 6. Check bank amount
        # ----------------------------------------------------

        if actual_settlement_amount != bank_amount:

            results.append({

                "payment_id":
                    payment_id,

                "gross_amount":
                    gross_amount,

                "payment_date":
                    payment_date,

                "settlement_id":
                    settlement_id,

                "settlement_date":
                    settlement_date,

                "fee":
                    fee,

                "tax":
                    tax,

                "expected_net_amount":
                    expected_net_amount,

                "settlement_net_amount":
                    actual_settlement_amount,

                "bank_amount":
                    bank_amount,

                "refund_amount":
                    refund_amount,

                "delay_days":
                    delay_days,

                "status":
                    "EXCEPTION",

                "reason":
                    "BANK_AMOUNT_MISMATCH"
            })

            continue

        # ----------------------------------------------------
        # 7. Everything passed
        # ----------------------------------------------------

        results.append({

            "payment_id":
                payment_id,

            "gross_amount":
                gross_amount,

            "payment_date":
                payment_date,

            "settlement_id":
                settlement_id,

            "settlement_date":
                settlement_date,

            "fee":
                fee,

            "tax":
                tax,

            "expected_net_amount":
                expected_net_amount,

            "settlement_net_amount":
                actual_settlement_amount,

            "bank_amount":
                bank_amount,

            "refund_amount":
                refund_amount,

            "delay_days":
                delay_days,

            "status":
                "AUTO_RECONCILED",

            "reason":
                "ALL_CHECKS_PASSED"
        })

    # ========================================================
    # SAVE RESULTS
    # ========================================================

    results_df = pd.DataFrame(results)

    output_path = (
        OUTPUT_DIR
        / "reconciliation_results.csv"
    )

    results_df.to_csv(
        output_path,
        index=False
    )

    print(
        f"Reconciliation complete."
    )

    print(
        f"Total records: {len(results_df)}"
    )

    print(
        "\nStatus counts:"
    )

    print(
        results_df["status"].value_counts()
    )

    print(
        "\nException reasons:"
    )

    print(
        results_df[
            results_df["status"] == "EXCEPTION"
        ]["reason"].value_counts()
    )

    print(
        f"\nResults saved to:"
        f"\n{output_path}"
    )

    return results_df


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    reconcile()

Reconciliation complete.
Total records: 100

Status counts:
status
EXCEPTION          82
AUTO_RECONCILED    18
Name: count, dtype: int64

Exception reasons:
reason
SETTLEMENT_DELAY                 36
REFUND_REQUIRES_INVESTIGATION    17
BANK_AMOUNT_MISMATCH             16
MISSING_BANK_RECORD              13
Name: count, dtype: int64

Results saved to:
C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot\Output\reconciliation_results.csv


In [6]:
if __name__ == "__main__":

    results = reconcile()

    print("\nReconciliation Complete!\n")

    print(
        results["status"].value_counts()
    )

    print("\nException Reasons:\n")

    print(
        results[
            results["status"] == "EXCEPTION"
        ]["reason"].value_counts()
    )

Reconciliation complete.
Total records: 100

Status counts:
status
EXCEPTION          82
AUTO_RECONCILED    18
Name: count, dtype: int64

Exception reasons:
reason
SETTLEMENT_DELAY                 36
REFUND_REQUIRES_INVESTIGATION    17
BANK_AMOUNT_MISMATCH             16
MISSING_BANK_RECORD              13
Name: count, dtype: int64

Results saved to:
C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot\Output\reconciliation_results.csv

Reconciliation Complete!

status
EXCEPTION          82
AUTO_RECONCILED    18
Name: count, dtype: int64

Exception Reasons:

reason
SETTLEMENT_DELAY                 36
REFUND_REQUIRES_INVESTIGATION    17
BANK_AMOUNT_MISMATCH             16
MISSING_BANK_RECORD              13
Name: count, dtype: int64


In [7]:
# Payment Date → Settlement Date
# 1-3 days = normal
# 7-14 days = delayed

In [8]:
def load_data():

    payments = pd.read_csv(
        DATA_DIR / r"C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot\Data\payments.csv",
        parse_dates=["payment_date"]
    )

    settlements = pd.read_csv(
        DATA_DIR / r"C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot\Data\settlements.csv",
        parse_dates=["settlement_date"]
    )

    bank_records = pd.read_csv(
        DATA_DIR / "bank_records.csv",
        parse_dates=["credit_date"]
    )

    refunds = pd.read_csv(
        DATA_DIR / "refunds.csv",
        parse_dates=["refund_date"]
    )

    return payments, settlements, bank_records, refunds

In [9]:
results[
    results["reason"] == "BANK_AMOUNT_MISMATCH"
].head()

,payment_id,gross_amount,payment_date,settlement_id,settlement_date,fee,tax,expected_net_amount,settlement_net_amount,bank_amount,refund_amount,delay_days,status,reason
9,pay_0010,1500,2026-08-31,set_0010,2026-09-01,30.0,5.4,1464.6,1464.6,964.6,NaN,1,EXCEPTION,BANK_AMOUNT_MISMATCH
11,pay_0012,15000,2026-08-07,set_0012,2026-08-10,300.0,54.0,14646.0,14646.0,14146.0,NaN,3,EXCEPTION,BANK_AMOUNT_MISMATCH
18,pay_0019,10000,2026-08-04,set_0019,2026-08-07,200.0,36.0,9764.0,9764.0,9514.0,NaN,3,EXCEPTION,BANK_AMOUNT_MISMATCH
33,pay_0034,5000,2026-08-27,set_0034,2026-08-28,100.0,18.0,4882.0,4882.0,4382.0,NaN,1,EXCEPTION,BANK_AMOUNT_MISMATCH
37,pay_0038,500,2026-08-09,set_0038,2026-08-11,10.0,1.8,488.2,488.2,238.2,NaN,2,EXCEPTION,BANK_AMOUNT_MISMATCH


In [11]:
import pandas as pd

results = pd.read_csv(
    r"C:\Users\anshu\OneDrive\Desktop\prog\Projects\reconpilot\Tests\reconciliation_results.csv"
)

print(
    results[
        results["payment_id"] == "pay_0093"
    ].T
)

                                        92
payment_id                        pay_0093
gross_amount                         25000
payment_date                    2026-08-07
settlement_id                     set_0093
settlement_date                 2026-08-09
fee                                  500.0
tax                                   90.0
expected_net_amount                    NaN
settlement_net_amount              24410.0
bank_amount                            NaN
refund_amount                          NaN
delay_days                               2
status                           EXCEPTION
reason                 MISSING_BANK_RECORD
